# 01 · Training, step by step (FSDR V4.0)

**Training** prepares the full customer base of one run date *D*. It validates the data, builds the features,
assigns the segments, and decides the two actions of every line. The scoring notebook (`02_scoring_step_by_step.ipynb`) then reads those results.

| Step | What it does | Writes |
|---|---|---|
| 0 | (optional) dummy data for tests | `data/training/<D>/`, `data/scoring/<D>/` |
| 1 | look at the two input folders (5 sub-folders each) | — |
| 2 | **T0** validation, completeness, Parquet cache | `data/runs/<D>/train/T0_load/`, `data/cache/<D>/` |
| 3 | look at the cache (16 partitions per table) | — |
| 4 | **T2** features B-01 … B-12 for the whole base | `data/runs/<D>/train/T2_features/` |
| 5 | **T3** S-rule segments, bands, micro-segments, week-0 history, `action_original` + `action_agent` | `data/runs/<D>/train/T3_segments/` |
| 6 | summary of the training steps | — |

**Folders of a run date (V4.0):**

```
data/training/<D>/   usage/ models/ subscription/ catalog/ others/     the base, history, model scores
data/scoring/<D>/    usage/ models/ subscription/ catalog/ others/     this week's scoring tables
                     target1/a_constraint/ target1/b_free/ target2/a_constraint/ target2/b_free/ control/
```

T0 reads the two folders **as one run**. A table filed on the wrong side is still loaded, with a warning.

**How to use:** set the run date in the next cell, then run the cells one by one (Shift+Enter). Read the explanation above each step.
Every step is **resumable**: a step already done with the same inputs is skipped, and `FORCE = True` re-runs it.

In [ ]:
# --- settings -------------------------------------------------------------------
RUN_DATE = "2026-10-22"     # the run date D (a Thursday in production)
GENERATE = True             # True: write dummy data for D first (tests); False: use the exports you copied
N_LINES = 3000              # dummy lines (2000 = 30 s; 1,000,000 = the production size, ~16 min to generate)
WORKERS = 2                 # partitions in parallel for T2 / T3 (2 on a laptop, 8 on a big machine)
FORCE = False               # True: re-run every step even if its inputs did not change
PREVIEW_ROWS = 8
# --- paths and helpers (do not edit) ---------------------------------------------
import json, os, sys
from datetime import date
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:      # works from notebooks/ or the repo root
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from nbx.fsdr import config, csvstd, steps

pd.set_option("display.max_columns", 80, "display.width", 220, "display.max_colwidth", 60)
D = date.fromisoformat(RUN_DATE)
TRAINING, SCORING = config.training_folder(RUN_DATE), config.scoring_folder(RUN_DATE)
RUN, CACHE, OUTPUT = config.FSDR_RUNS_DIR / RUN_DATE, config.FSDR_CACHE_DIR / RUN_DATE, config.output_folder(RUN_DATE)


def show(obj, n=PREVIEW_ROWS, title=None):
    """A table (DataFrame, list of dicts, dict or CSV / Parquet / csv.gz path), first n rows."""
    if isinstance(obj, (str, Path)):
        p = Path(obj)
        title = title or (str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else str(p))
        if not p.exists() or p.stat().st_size == 0:
            print(f"{title}: empty or not written")
            return
        try:
            obj = (csvstd.read_csv(p) if p.name.endswith(".csv.gz") else pd.read_parquet(p) if p.suffix == ".parquet"
                   else pd.read_csv(p) if p.suffix == ".csv" else json.loads(p.read_text(encoding="utf-8")))
        except pd.errors.EmptyDataError:
            print(f"{title}: no rows")
            return
    if isinstance(obj, dict):
        obj = pd.DataFrame([{"key": k, "value": v if not isinstance(v, (dict, list)) else json.dumps(v, default=str)[:150]}
                            for k, v in obj.items()])
    if isinstance(obj, list):
        obj = pd.DataFrame(obj)
    if title:
        display(Markdown(f"**{title}** · {len(obj):,} rows x {obj.shape[1]} columns"))
    display(obj.head(n))


def tree(folder, depth=3, skip_schemas=True):
    """The files of a folder, as a tree with sizes."""
    folder = Path(folder)
    if not folder.exists():
        print(f"{folder}: does not exist yet")
        return
    print(folder)
    for p in sorted(folder.rglob("*")):
        rel = p.relative_to(folder)
        if len(rel.parts) > depth or (skip_schemas and p.name.endswith(".schema.json")):
            continue
        pad = "    " * (len(rel.parts) - 1)
        print(f"{pad}{p.name}{'/' if p.is_dir() else f'  ({p.stat().st_size / 1e3:,.1f} kB)'}")


def step_status(phase_dir, name):
    meta = steps.read_step(Path(phase_dir) / name)
    print(f"{name}: {meta.get('status', 'not run')}  rows {meta.get('rows', {})}  {meta.get('seconds', '-')} s"
          + (f"  reason: {meta['reason']}" if meta.get("reason") else ""))
    return meta


print(f"run date : {RUN_DATE}")
for name, p in (("training", TRAINING), ("scoring", SCORING), ("runs", RUN), ("cache", CACHE), ("output", OUTPUT)):
    print(f"{name:9s}: {p}  ({'found' if p.exists() else 'not yet'})")
print(f"offers per line: {config.RECO_OFFERS}  (FSDR_RECO_OFFERS)")

## Step 0 · Dummy data (optional)

The generator writes the 47 tables of run date *D* as `.csv.gz`, the FSDR CSV standard: gzip, UTF-8, IDs kept as text.

- The training tables (base, usage history, model scores) go to `data/training/<D>/`.
- The scoring tables of the week go to `data/scoring/<D>/`: `subscriptions_tg`, `usage_data_voice_payg_tg`, `sub_btl_3months`, `targeting_auto`, the catalogue, the scripts, the conflicts, consent, and the `*_score` tables.
- `btl_catalog` is written to both sides.

The manifest says `source: generator`, so real exports are never overwritten silently. With real data, set `GENERATE = False` and copy your files into the same folders.

In [ ]:
if GENERATE:
    from nbx.fsdr.synth import generator

    man = TRAINING / config.MANIFEST_FILE
    if TRAINING.exists() and any(TRAINING.iterdir()) and (not man.exists() or json.loads(man.read_text())["source"] != "generator"):
        raise SystemExit(f"{TRAINING} holds real data: set GENERATE = False")
    run = generator.generate(N_LINES, D, ["baseline"], config.GENERATOR_SEED, TRAINING, "csv.gz", "split")
    print(f"{len(run.tables)} tables generated for {N_LINES:,} lines")
else:
    print("GENERATE = False: using the files already in", TRAINING, "and", SCORING)

## Step 1 · The two input folders

Every table lives in one of the 5 sub-folders. Its side (training or scoring) is set by `config.SCORING_TABLES`.
The table below lists, for each table of the data contract:
- the side and folder it was found in;
- its format and size;
- its row count from the manifest.

**Check:** `misfiled` should be empty, and so should `duplicates` (two files for one table block T0).

In [ ]:
from nbx.fsdr import schemas
from nbx.fsdr.datasource import JsonSource

src = JsonSource(TRAINING)
man = src.manifest()
rows = []
for t, s in schemas.TABLES.items():
    p = src.path(t)
    rows.append({"side": src.side_of.get(t, "-"), "folder": s.folder, "table": t, "present": p is not None,
                 "expected side": config.table_side(t), "MB": round(p.stat().st_size / 1e6, 3) if p else 0,
                 "rows (manifest)": (man.get("row_counts") or {}).get(t)})
inputs = pd.DataFrame(rows).sort_values(["side", "folder", "table"])
print(f"twins: {', '.join(f'{k} = {v}' for k, v in src.twins.items())}")
print(f"tables found: {inputs.present.sum()} / {len(inputs)} | misfiled: {src.misfiled() or 'none'} | "
      f"duplicates: {src.duplicate_files or 'none'}")
display(inputs.groupby(["side", "folder"]).agg(tables=("table", "count"), MB=("MB", "sum")).round(2))
show(inputs, n=60)

## Step 2 · T0: validation, completeness, Parquet cache (① data agent)

T0 runs before anything else and does three things.

1. **Validation** against the data contract: columns, types, keys, value ranges, date windows, the `.csv.gz` format and the folders.
   - A *blocking* issue stops the run.
   - A *warning* is reported and the run continues.
2. **Completeness**: are the tables the pipeline needs all present?
   - Week 0 needs the `*_auto` tables; a weekly run needs the `*_score` tables.
   - A missing optional table (sample, consent) falls back to a documented default.
3. **Cache**: every table is written to Parquet in 16 partitions, by a hash of `ACCESS_METHOD_ID`.
   - Every later step reads this cache, never the `.csv.gz` again, so 1M lines fit in memory partition by partition.

Outputs: `validation_report.md / .json`, `completeness.csv`, `cache_manifest.json`, `summary.json`.

In [ ]:
from nbx.fsdr.contract import data_agent

t0 = data_agent.run_t0(TRAINING, run_date=D, force=FORCE)
print(f"T0_load {t0['status'].upper()}{' (skipped: same inputs)' if t0.get('skipped') else ''} -- pipeline {t0['pipeline']}, "
      f"{t0['tables']} tables, {t0['rows_in']:,} rows, validation {t0['validation']} {t0['issue_counts']}")
print("read as one run:", t0.get("training_source"), "+", t0.get("scoring_source"))
print()
print(t0["explanation"])

In [ ]:
T0 = RUN / "train" / "T0_load"
rep = json.loads((T0 / "validation_report.json").read_text(encoding="utf-8"))
issues = pd.DataFrame(rep["issues"])
if len(issues):
    display(issues.groupby(["level", "check"]).size().rename("issues").reset_index())
    show(issues[["level", "check", "table", "column", "message"]], n=20, title="validation issues")
else:
    print("validation: no issue")
show(T0 / "completeness.csv", n=50)

## Step 3 · The Parquet cache

Each table is split into 16 partitions. A line is always in the same partition across tables, so T2 / T3 can
join the tables of one partition without reading the whole base. **Check:** `verify_cache` returns no problem.

In [ ]:
from nbx.fsdr.contract.cache import CacheSource, verify_cache

cache = CacheSource(D)
cm = cache.manifest()
tables = pd.DataFrame([{"table": t, **{k: v for k, v in m.items() if k in ("rows", "partitioned", "bytes")}}
                       for t, m in cm["tables"].items()])
print(f"cache {CACHE}: {cm['partitions']} partitions, {cm['bytes'] / 1e6:,.1f} MB, built in {cm['seconds']} s")
print("verify_cache:", verify_cache(D) or "OK")
show(tables.sort_values("rows", ascending=False), n=50)

## Step 4 · T2: features of the whole base (B-01 … B-12)

T2 works partition by partition, with `WORKERS` partitions in parallel. For every eligible line it builds:

- **Eligibility** (B-01 … B-03):
  - active prepaid lines;
  - valid 8-digit MSISDN;
  - not an internal / employee plan;
  - duplicates removed.
- **Profile**: age, tenure, nationality group (Oman, India, Bangladesh, Pakistan, others), region, device segment.
- **Usage and revenue**: weekly (w1 … w4) and monthly (m1 … m3) activity, BTL and pay-as-you-go spend, the trends between periods.
- **Model scores** (B-05): the 11 model tables (churn, appetency, …). A missing score gets the median of the line's nationality group.

The anchor decides which tables are read: week 0 reads the `*_auto` tables (`anchor = T`), a weekly run reads the `*_score` tables.

Outputs:
- `base_features/part-NN.parquet` (1 file per partition);
- `feature_summary.csv` (count, nulls, mean, min, max per feature);
- `stats.json` (the medians and the ARPU tertiles that T3 uses);
- `base_features_preview.csv`.

In [ ]:
from nbx.fsdr.pipelines import training

t2 = training.run_t2(D, workers=WORKERS, force=FORCE)
print(f"T2_features DONE{' (skipped)' if t2.get('skipped') else ''}: {t2['lines']:,} eligible lines of {t2['base_rows']:,}, "
      f"{t2['features']} features, anchor {t2['anchor']} ({t2['tables']})")
show({"eligibility filters": t2["eligible_filters"], "missing model scores": t2["missing_share"],
      "age source": t2["age_source"], "tenure source": t2["tenure_source"], "ARPU tertiles": t2["arpu_tertiles"],
      "msisdn in two partitions (must be 0)": t2["msisdn_in_two_partitions"]}, title="T2 summary")

In [ ]:
T2 = RUN / "train" / "T2_features"
fs = pd.read_csv(T2 / "feature_summary.csv")
fs["null_share"] = (fs["nulls"] / fs["count"].where(fs["count"] > 0)).round(4)
show(fs.sort_values("null_share", ascending=False), n=15, title="features with the most missing values")
stats = json.loads((T2 / "stats.json").read_text(encoding="utf-8"))
show(pd.DataFrame(stats["medians_by_nationality"]).T.round(4), n=10, title="B-05 medians by nationality (missing scores)")
show(T2 / "base_features_preview.csv", n=5)

## Step 5 · T3: segments, micro-segments and the two actions

For every line of T2:

- **Segment**: the S-rules (in order, the first that matches wins) give one of the 14 segments (Retention, RGS, Onboarding, Nursering).
  - The Nursering lines are **excluded** (B-02): they are never contacted.
- **Bands**:
  - age (young / middle / senior);
  - tenure;
  - value band (ARPU tertiles from T2);
  - usage type (voice / data / both).
- **Micro-segment** = nationality | age band | tenure band. It is used for the balance of the panel and the median anchor.
- **Week-0 history**: what the production engine targeted last week (`W0_*`), and who took an offer.
- **Two actions per line** (one of the 7 actions):
  - `ACTION_ORIGINAL`: the production dictionary (segment -> action). Target 1 uses it.
  - `ACTION_AGENT`: the agent dictionary, versioned and checked against the previous version. Target 2 uses it.

Outputs:
- `segments/part-NN.parquet`;
- `segment_counts.csv`, `microsegment_counts.csv`, `action_counts.csv`, `crosswalk.csv`;
- `priors.json` (B-10 action priors) and the agent dictionary `.json`.

In [ ]:
t3 = training.run_t3(D, workers=WORKERS, force=FORCE)
print(f"T3_segments DONE{' (skipped)' if t3.get('skipped') else ''}: {t3['lines']:,} lines, {t3['targetable']:,} targetable, "
      f"{t3['excluded_nursering']:,} Nursering excluded; {t3['segments_non_empty']} segments, "
      f"{t3['micro_segments_non_empty']} / {t3['micro_segments_total']} micro-segments; agent dictionary {t3['agent_dictionary']}")
T3 = RUN / "train" / "T3_segments"
seg = pd.read_csv(T3 / "segment_counts.csv")
show(seg, n=20)
ax = seg.set_index("SEGMENT")["lines"].sort_values().plot.barh(figsize=(8, 5), title="lines per segment")
plt.tight_layout(); plt.show()

In [ ]:
acts = pd.read_csv(T3 / "action_counts.csv")
pivot = acts.pivot_table(index="ACTION_ORIGINAL", columns="ACTION_AGENT", values="lines", aggfunc="sum", fill_value=0)
display(Markdown(f"**action_original (rows) x action_agent (columns)** · same action on "
                 f"{t3['action_original_eq_agent']:.1%} of the lines"))
display(pivot)
show(T3 / "microsegment_counts.csv", n=10)
show(json.loads((T3 / "priors.json").read_text(encoding="utf-8")), title="B-10 action priors")
show(T3 / "segments_preview.csv", n=5)

## Step 6 · Training summary

Every step folder holds a `_step.json` with its status (done / blocked / failed), its rows, its duration and its
outputs. When all three steps are **done**, open `02_scoring_step_by_step.ipynb`.

In [ ]:
rows = [{"step": s["step"], "status": s["status"], "seconds": s["seconds"], "rows": s["rows"], "files": len(s["files"])}
        for s in steps.list_steps(RUN) if s["phase"] == "train"]
show(rows, n=20, title=f"training steps of {RUN_DATE}")
tree(RUN / "train", depth=1)